# pi-metaboqc Interactive Pipeline Tutorial

This notebook provides a step-by-step interactive guide to the `pi-metaboqc` 
metabolomics data quality control pipeline. 

By executing each cell sequentially, you can inspect the intermediate 
data structures and quality assessment (QA) reports generated at each 
stage of the workflow.

## Pipeline Architecture Overview
1. Dataset Construction
2. QA (Raw Data)
3. MV Filtering (Stage 1)
4. Signal Correction (Intra & Inter-batch)
5. QA (Intra & Inter-batch)
6. Quality Filtering (Stage 2)
7. QA (Post-Filtering)
8. Imputation
9. QA (Post-Imputation)
10. Normalization
11. Final QA

## Step 0: Environment Setup

First, we will import the core modules of `pi-metaboqc`, configure the input data paths, and create a dedicated output directory for the results of this notebook.

In [ ]:
# -------------------------------------------------------------------------
# Step 0: Environment Setup and Parameter Loading
# -------------------------------------------------------------------------
import os
import sys
import pandas as pd
from loguru import logger
from typing import Dict, Any, Tuple

# # Append the src directory to system path to import pimqc modules
# sys.path.append(os.path.abspath(os.path.join("..", "src")))

import pimqc.io_utils as iu
import pimqc.plot_utils as pu
from pimqc.dataset_builder import build_dataset
from pimqc.data_quality_assessment import MetaboIntQA
from pimqc.invalid_feature_sample_filtering import MetaboIntFLTR
from pimqc.data_signal_correction import MetaboIntSC
from pimqc.imputation import MetaboIntImputer
from pimqc.normalization import MetaboIntNorm

# Define standard directories
DATA_DIR: str = os.path.join("..", "src", "pimqc", "data")
OUTPUT_DIR: str = os.path.join(".", "tutorial_output")
iu._check_dir_exists(dir_path=OUTPUT_DIR, handle="makedirs")

# Load SSOT pipeline parameters
PARAMS_PATH: str = os.path.join(DATA_DIR, "pipeline_parameters.json")
params: Dict[str, Any] = iu._load_json_file(input_file_path=PARAMS_PATH)

# Load raw matrices
meta_df: pd.DataFrame = pd.read_csv(
    os.path.join(DATA_DIR, "project_meta.csv"), header=[0]
)
int_df: pd.DataFrame = pd.read_csv(
    os.path.join(DATA_DIR, "project_intensity.csv"), index_col=[0], header=[0]
)

logger.info("Environment initialized successfully.")

## Step 1: Dataset Construction
We will merge the raw intensity dataframe with the metadata mapping to 
build the standardized `MetaboInt` base object. This multi-indexed dataframe 
ensures analytical features and sample metadata are strictly aligned.

In [ ]:
# -------------------------------------------------------------------------
# Step 1: Build Standardized Dataset
# -------------------------------------------------------------------------
step1_dir: str = os.path.join(OUTPUT_DIR, "01_Raw_Data")

raw_data = build_dataset(
    meta_info=meta_df,
    int_df=int_df,
    pipeline_params=params,
    mode=params.get("MetaboInt", {}).get("mode", "POS"),
    batch=params.get("MetaboInt", {}).get("batch", "Batch"),
    sample_type=params.get(
        "MetaboInt", {}).get("sample_type", "Sample Type"
    ),
    bio_group=params.get(
        "MetaboInt", {}).get("bio_group", "Bio Group"
    ),
    sample_name=params.get(
        "MetaboInt", {}).get("sample_name", "Sample Name"
    ),
    inject_order=params.get(
        "MetaboInt", {}).get("inject_order", "Inject Order"
    ),
    output_dir=step1_dir
)

logger.info(f"Raw MetaboInt object built. Shape: {raw_data.shape}")

## Step 2: Quality Assessment (Raw Data)
Before any mathematical transformation, we perform an initial QA to evaluate 
the baseline condition of the untargeted metabolomics data.

In [ ]:
# -------------------------------------------------------------------------
# Step 2: Initial QA
# -------------------------------------------------------------------------
step2_dir: str = os.path.join(OUTPUT_DIR, "02_QA_Raw_Data")

qa_raw = MetaboIntQA(data=raw_data, pipeline_params=params)
qa_raw.execute_qa(output_dir=step2_dir)

logger.info("Initial QA completed.")

## Step 3: High Missing Value (MV) Feature Filter (Stage 1)
We drop features containing an unacceptably high proportion of missing 
values based on the defined `mv_group_tol` biological override rule.

In [ ]:
# -------------------------------------------------------------------------
# Step 3: Stage 1 Filtering
# -------------------------------------------------------------------------
fltr_stg1 = MetaboIntFLTR(data=raw_data, pipeline_params=params)
filtered_data = fltr_stg1.execute_mv_fltr()

logger.info(f"Post-Stage 1 data shape: {filtered_data.shape}")

## Step 4: Signal Drift & Batch Effect Correction
Using Quality Control (QC) samples, we build regression models (e.g., LOESS, 
Random Forest, SVR) to estimate and correct analytical signal drift 
across injection sequences.

In [ ]:
# -------------------------------------------------------------------------
# Step 4: Signal Correction
# -------------------------------------------------------------------------
step4_dir: str = os.path.join(OUTPUT_DIR, "04_Corrected_Data")

sc_engine = MetaboIntSC(data=filtered_data, pipeline_params=params)
intra_data, inter_data = sc_engine.execute_sc(output_dir=step4_dir)

logger.info(f"Intra-batch SC shape: {intra_data.shape}")
logger.info(f"Inter-batch SC shape: {inter_data.shape}")

## Step 5 & 6: Quality Assessment on Corrected Data
We assess the effectiveness of the signal correction algorithms by examining 
the intra-batch and inter-batch states separately.

In [ ]:
# -------------------------------------------------------------------------
# Step 5 & 6: QA on Intra and Inter-batch Corrected Data
# -------------------------------------------------------------------------
step5_dir: str = os.path.join(OUTPUT_DIR, "05_QA_Intra_SC")
step6_dir: str = os.path.join(OUTPUT_DIR, "06_QA_Inter_SC")

qa_intra = MetaboIntQA(data=intra_data, pipeline_params=params)
qa_intra.execute_qa(output_dir=step5_dir)

qa_inter = MetaboIntQA(data=inter_data, pipeline_params=params)
qa_inter.execute_qa(output_dir=step6_dir)

logger.info("Post-Correction QA reports generated.")

## Step 7: Low Quality Feature Filter (Stage 2)
Features are evaluated against Blank sample ratios and QC Relative Standard 
Deviation (RSD). Missing indices are passed explicitly as current features 
default to MAR behavior prior to imputation.

In [ ]:
# -------------------------------------------------------------------------
# Step 7: Stage 2 Filtering
# -------------------------------------------------------------------------
fltr_stg2 = MetaboIntFLTR(data=inter_data, pipeline_params=params)

idx_mar: pd.Index = inter_data.index
idx_mnar: pd.Index = pd.Index([])

quality_filtered_data = fltr_stg2.execute_quality_fltr(
    idx_mar=idx_mar, idx_mnar=idx_mnar
)

logger.info(f"Post-Stage 2 data shape: {quality_filtered_data.shape}")

## Step 8: Quality Assessment on Filtered Data

In [ ]:
# -------------------------------------------------------------------------
# Step 8: QA Post-Filtering
# -------------------------------------------------------------------------
step8_dir: str = os.path.join(OUTPUT_DIR, "08_QA_Filtered_Data")

qa_fltr = MetaboIntQA(data=quality_filtered_data, pipeline_params=params)
qa_fltr.execute_qa(output_dir=step8_dir)

## Step 9: Missing Value Imputation
We address remaining structural missing values using probabilistic logic, 
Random Forest, or traditional k-Nearest Neighbors (KNN).

In [ ]:
# -------------------------------------------------------------------------
# Step 9: Missing Value Imputation & Evaluation
# -------------------------------------------------------------------------
from pimqc.imputation import MetaboIntImputer
import os

step9_dir = os.path.join(OUTPUT_DIR, "09_Imputed_Data")

# 1. Initialize imputation engine
imp_engine = MetaboIntImputer(
    quality_filtered_data, 
    pipeline_params=params
)

# 2. Smart Routing: Intercept 'auto' mode to prevent duplicate benchmarking
target_method = imp_engine.attrs.get("method", "auto")
if target_method.lower() == "auto":
    # Trigger the benchmark once and lock in the optimal method
    target_method = imp_engine._auto_select_method(mask_ratio=0.05)
    
    # Overwrite the 'auto' attribute to prevent downstream re-triggering
    imp_engine.attrs["method"] = target_method 

# 3. Execute Simulation (Calculates NRMSE and silently saves scatter plot)
sim_nrmse = imp_engine.execute_nrmse_simulation(
    method=target_method, output_dir=step9_dir
)

# 4. Execute Imputation (Silently saves imputed CSV, KDE, and PCA plots)
imputed_data = imp_engine.execute_imputation(
    method=target_method, output_dir=step9_dir
)

logger.success(
    f"Imputation completed using '{target_method}'. Shape: {imputed_data.shape}. "
    f"All data and analytical plots saved to {step9_dir}."
)

## Step 10: Quality Assessment on Imputed Data

In [ ]:
# -------------------------------------------------------------------------
# Step 10: QA Post-Imputation
# -------------------------------------------------------------------------
step10_dir: str = os.path.join(OUTPUT_DIR, "10_QA_Imputed_Data")

qa_imp = MetaboIntQA(data=imputed_data, pipeline_params=params)
qa_imp.execute_qa(output_dir=step10_dir)

## Step 11: Data Normalization
Features are scaled and normalized (e.g., PQN, VSN) to remove remaining 
systematic bias and stabilize variance across the detection range.

In [ ]:
# -------------------------------------------------------------------------
# Step 11: Normalization
# -------------------------------------------------------------------------
step11_dir: str = os.path.join(OUTPUT_DIR, "11_Normalized_Data")

# 1. Initialize normalization engine (norm_engine serves as the Raw Data)
norm_engine = MetaboIntNorm(imputed_data, pipeline_params=params)
# 2. Execute normalization (this automatically saves comparison PDFs to disk)
normalized_data = norm_engine.execute_norm(output_dir=step11_dir)

logger.info(f"Normalized dataset shape: {normalized_data.shape}")

## Step 12: Final Quality Assessment
The fully processed and refined dataset is evaluated one last time, 
confirming its readiness for downstream statistical learning and 
biomarker discovery.

In [ ]:
# -------------------------------------------------------------------------
# Step 12: Final QA
# -------------------------------------------------------------------------
step12_dir: str = os.path.join(OUTPUT_DIR, "12_QA_Normalized_Data")

qa_final = MetaboIntQA(data=normalized_data, pipeline_params=params)
qa_final.execute_qa(output_dir=step12_dir)

logger.success("Pipeline tutorial completed successfully.")